In [1]:
!wget https://github.com/HanjieChen/ChallengeClinicalQA/raw/main/medbullets/medbullets_op4.csv


--2025-03-30 19:08:39--  https://github.com/HanjieChen/ChallengeClinicalQA/raw/main/medbullets/medbullets_op4.csv
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/HanjieChen/ChallengeClinicalQA/main/medbullets/medbullets_op4.csv [following]
--2025-03-30 19:08:40--  https://raw.githubusercontent.com/HanjieChen/ChallengeClinicalQA/main/medbullets/medbullets_op4.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1209737 (1.2M) [text/plain]
Saving to: ‘medbullets_op4.csv.2’

medbullets_op4.csv. 100%[===================>]   1.15M  --.-KB/s    in 0.04s   

2025-03-30 19:08:40 (30.1 MB/s) - ‘medbullets

In [25]:
import requests

# URLs of the datasets
url_op4 = "https://github.com/HanjieChen/ChallengeClinicalQA/raw/main/medbullets/medbullets_op4.csv"
# Download and save the files
with open("medbullets_op4.csv", "wb") as file:
    file.write(requests.get(url_op4).content)


In [26]:
import pandas as pd

# Load the CSV file
df_op4 = pd.read_csv("medbullets_op4.csv")

# Count duplicates before removing
num_duplicates = df_op4.duplicated(subset=['question']).sum()

# Remove duplicate rows based on the 'question' column, keeping the first occurrence
df_op4 = df_op4.drop_duplicates(subset=['question'], keep='first')

# Display the number of duplicate rows found
print(f"Number of duplicate rows removed: {num_duplicates}")

# Display the first few rows after removing duplicates
print(df_op4.columns)
print(df_op4.head())
print(len(df_op4))

Number of duplicate rows removed: 10
Index(['link', 'question', 'opa', 'opb', 'opc', 'opd', 'answer_idx', 'answer',
       'explanation'],
      dtype='object')
                                               link  \
0                            https://bit.ly/47KxMQs   
1                            https://bit.ly/3GZtkBx   
2  https://step2.medbullets.com/testview?qid=217175   
3                            https://bit.ly/40qTdn2   
4                            https://bit.ly/3QOJp1p   

                                            question  \
0  A 42-year-old woman is enrolled in a randomize...   
1  A 9-year-old girl presents to the emergency de...   
2  A 1-year-old girl is brought to a neurologist ...   
3  A 17-year-old boy presents to his primary care...   
4  A 55-year-old woman is brought to the emergenc...   

                                              opa  \
0  AV node > ventricles > atria > Purkinje fibers   
1                                 Gastroenteritis   
2           

In [4]:
import openai

def generate_direct_prediction(question, opa, opb, opc, opd):
    """
    Uses GPT-4o to predict the correct answer from a multiple-choice question using separate fields for question and answer choices.
    Returns only the predicted answer in the format: 'B: Femoral artery murmur'
    """
    prompt = f"""
The following is a medical multiple-choice question. It includes a clinical vignette followed by four answer options labeled A, B, C, and D.

Question:
{question}

Choices:
A. {opa}
B. {opb}
C. {opc}
D. {opd}

Your task:
- Select the best answer.
- Respond strictly in the following format: [Letter]: [Answer Text] (e.g., B: Femoral artery murmur)
- Do not explain. Do not repeat the question.
"""

    try:
        client = openai.OpenAI()

        response = client.chat.completions.create(
            model="o3-mini",
            messages=[{"role": "user", "content": prompt}]
        )

        content = response.choices[0].message.content.strip()
        return content

    except Exception as e:
        return "Error"


In [5]:
# Apply the function only to the first 5 rows
df_op4.loc[:4, "gpt_direct_prediction"] = df_op4.loc[:4].apply(
    lambda row: generate_direct_prediction(
        row["question"], row["opa"], row["opb"], row["opc"], row["opd"]
    ),
    axis=1
)
print(df_op4[["question", "gpt_direct_prediction"]].head(5))


                                            question  \
0  A 42-year-old woman is enrolled in a randomize...   
1  A 9-year-old girl presents to the emergency de...   
2  A 1-year-old girl is brought to a neurologist ...   
3  A 17-year-old boy presents to his primary care...   
4  A 55-year-old woman is brought to the emergenc...   

                               gpt_direct_prediction  
0  C: Purkinje fibers > atria > ventricles > AV node  
1                       B: Intentional contamination  
2                             A: Cardiac rhabdomyoma  
3       A: Continue current therapy for 1 more month  
4                           A: Cerebral salt wasting  


In [6]:
df_op4["gpt_direct_prediction"] = df_op4.apply(
    lambda row: generate_direct_prediction(
        row["question"], row["opa"], row["opb"], row["opc"], row["opd"]
    ),
    axis=1
)

# print(df_op4[["question", "gpt_direct_prediction"]].head(5))

In [7]:
print(df_op4[["question", "gpt_direct_prediction"]].head(15))

                                             question  \
0   A 42-year-old woman is enrolled in a randomize...   
1   A 9-year-old girl presents to the emergency de...   
2   A 1-year-old girl is brought to a neurologist ...   
3   A 17-year-old boy presents to his primary care...   
4   A 55-year-old woman is brought to the emergenc...   
5   A 2-week-old boy is evaluated by his pediatric...   
6   A 1-month-old girl presents to her pediatricia...   
7   A newborn boy is evaluated in the hospital nur...   
8   A 55-year-old male bodybuilder presents to the...   
10  A 57-year-old man presents to the emergency de...   
11  A 39-year-old man presents to his doctor for a...   
12  An 84-year-old man presents to the physician w...   
14  A 51-year-old man presents for his annual well...   
15  A 70-year-old woman is brought to the emergenc...   
16  A 72-year-old man presents to his primary care...   

                                gpt_direct_prediction  
0   C: Purkinje fibers > atria 

In [38]:
# Extract the predicted letter from GPT output (e.g., 'A' from 'A: Cardiac rhabdomyoma')
df_op4["gpt_letter"] = df_op4["gpt_direct_prediction"].astype(str).str.strip().str[0]

# Clean answer_idx to ensure it's a string and uppercase
df_op4["answer_letter"] = df_op4["answer_idx"].astype(str).str.strip().str.upper()

# Compare the predicted letter to the answer letter
df_op4["gpt_letter_match"] = df_op4.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Compute accuracy
correct_count = (df_op4["gpt_letter_match"] == "Correct").sum()
total_count = df_op4["gpt_letter_match"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

# Print results
print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")


Letter-Based Correct Predictions: 224
Total Predictions Compared: 298
Letter Match Accuracy: 75.17%


## With Bullet Points

In [27]:
import pandas as pd
import re

# Function to process and format question, and count numbered sentences
def process_question(row):
    # Split the question text into sentences
    sentences = re.split(r'(?<=[.!?])\s+', str(row['question']).strip())
    
    if len(sentences) > 1:
        # Number all but the last sentence
        numbered_sentences = [f"{i+1}. {sentence.strip()}" for i, sentence in enumerate(sentences[:-1])]
        formatted_question = "\n".join(numbered_sentences)
        
        # Add a blank line and include the final question sentence without numbering
        formatted_question += f"\n\n{sentences[-1].strip()}"
        num_sentences = len(sentences) - 1
    else:
        formatted_question = sentences[0].strip()
        num_sentences = 0

    # Append formatted answer choices
    options = f"\n\nA. {row['opa'].strip()}\nB. {row['opb'].strip()}\nC. {row['opc'].strip()}\nD. {row['opd'].strip()}"

    return formatted_question + options, num_sentences

# Apply the function and split results into two columns
df_op4[['actual_question', 'number_sentences']] = df_op4.apply(
    lambda row: pd.Series(process_question(row)),
    axis=1
)

# Create bullet_question by replacing numbered steps with dashes
df_op4['bullet_question'] = df_op4['actual_question'] \
    .str.replace(r'^\d+\.\s+', '- ', regex=True) \
    .str.replace(r'\n\d+\.\s+', '\n- ', regex=True)

# Display preview
pd.set_option('display.max_colwidth', None)
print(df_op4[['question', 'actual_question', 'bullet_question', 'number_sentences']].head(2))
pd.reset_option('display.max_colwidth')

# Save the processed DataFrame to CSV
df_op4.to_csv("processed_medbullets_op4.csv", index=False)


In [28]:
print(len(df_op4))


298


## GPT 4o Results Direct Prediction

In [29]:
# prompt: use OpenAI GPT4o to extract first columns' paper title. My prompt: Please extract paper title from this sentence.

!pip uninstall -y openai
!pip install --upgrade openai

import openai
print(openai.__version__)
import pandas as pd

# Assuming 'senior_author' DataFrame is loaded and contains a column named 'Paper Title'

# Set your OpenAI API key
openai.# Replace with your actual API key

^C
Traceback (most recent call last):
  File "/data/healthy-ml/scratch/yuexing/anaconda3/bin/pip", line 7, in <module>
    from pip._internal.cli.main import main
  File "/data/healthy-ml/scratch/yuexing/anaconda3/lib/python3.9/site-packages/pip/_internal/cli/main.py", line 9, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/data/healthy-ml/scratch/yuexing/anaconda3/lib/python3.9/site-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/data/healthy-ml/scratch/yuexing/anaconda3/lib/python3.9/site-packages/pip/_internal/cli/main_parser.py", line 8, in <module>
    from pip._internal.cli import cmdoptions
  File "/data/healthy-ml/scratch/yuexing/anaconda3/lib/python3.9/site-packages/pip/_internal/cli/cmdoptions.py", line 23, in <module>
    from pip._internal.cli.parser import ConfigOptionParser
  File "/data/healthy-ml/scratch/yuexing/anaconda3/lib/python3.9/site-pa

In [30]:
import openai

def generate_direct_prediction(question_text):
    """
    Uses GPT-4o to predict the correct answer from a multiple-choice question embedded in a single string.
    Returns only the predicted answer in the format: 'B: Femoral artery murmur'
    """
    prompt = f"""
The following is a medical multiple-choice question. It includes a clinical vignette followed by four answer options labeled A, B, C, and D.

{question_text}

Your task:
- Select the best answer.
- Respond strictly in the following format: [Letter]: [Answer Text] (e.g., B: Femoral artery murmur)
- Do not explain. Do not repeat the question.
"""

    try:
        # Initialize OpenAI client
        client = openai.OpenAI(#Set the API key. See the how-to guide for further instructions
        )  # Set your API key
        # Generate response using GPT-4o
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature = 0
        )
        content = response.choices[0].message.content.strip()
        return content
    except Exception as e:
        return "Error"


In [31]:
import openai

def generate_direct_prediction(question_text):
    """
    Uses GPT-4o to predict the correct answer from a multiple-choice question embedded in a single string.
    Returns only the predicted answer in the format: 'B: Femoral artery murmur'
    """
    prompt = f"""
The following is a medical multiple-choice question. It includes a clinical vignette followed by four answer options labeled A, B, C, and D.

{question_text}

Your task:
- Select the best answer.
- Respond strictly in the following format: [Letter]: [Answer Text] (e.g., B: Femoral artery murmur)
- Do not explain. Do not repeat the question.
"""

    try:
        # Initialize OpenAI client
        client = openai.OpenAI(#Set the API key. See the how-to guide for further instructions
        )  # Set your API key
        # Generate response using GPT-4o
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature = 0
        )
        content = response.choices[0].message.content.strip()
        return content
    except Exception as e:
        return "Error"


In [32]:
# Example: first 5 rows for testing
df_subset = df_op4.head(298).copy()

# Apply GPT-4o direct prediction to the 'actual_question' column
df_subset["gpt4o_random1_direct_prediction"] = df_subset["actual_question"].apply(generate_direct_prediction)

# Show results
pd.set_option("display.max_colwidth", None)
display(df_subset[["actual_question", "gpt4o_random1_direct_prediction"]])


,actual_question,gpt4o_random1_direct_prediction
0,"1. A 42-year-old woman is enrolled in a randomized controlled trial to study cardiac function in the setting of several different drugs.\n2. She is started on verapamil and instructed to exercise at 50% of her VO2 max while several cardiac parameters are being measured.\n\nDuring this experiment, which of the following represents the relative conduction speed through the heart from fastest to slowest?\n\nA. AV node > ventricles > atria > Purkinje fibers\nB. Purkinje fibers > ventricles > atria > AV node\nC. Purkinje fibers > atria > ventricles > AV node\nD. Purkinje fibers > AV node > ventricles > atria",B: Purkinje fibers > ventricles > atria > AV node
1,"1. A 9-year-old girl presents to the emergency department with a fever and a change in her behavior.\n2. She presented with similar symptoms 6 weeks ago and was treated for an Escherchia coli infection.\n3. She also was treated for a urinary tract infection 10 weeks ago.\n4. Her mother says that last night her daughter felt ill, and her condition has been worsening.\n5. Her daughter experienced a severe headache and had a stiff neck.\n6. This morning she was minimally responsive, vomited several times, and produced a small amount of dark cloudy urine.\n7. The patient was born at 39 weeks and met all her developmental milestones.\n8. She is currently up to date on her vaccinations and did not have infections during early childhood.\n9. Her parents are divorced and her father has noted she does not seem to get sick when he takes care of her.\n10. Her temperature is 99.5°F (37.5°C), blood pressure is 60/35 mmHg, pulse is 190/min, respirations are 33/min, and oxygen saturation is 98% on room air.\n11. The patient is started on intravenous fluids, vasopressors, and broad-spectrum antibiotics.\n\nWhich of the following is the most appropriate underlying explanation for this patient's presentation?\n\nA. Gastroenteritis\nB. Intentional contamination\nC. Meningitis\nD. Urinary tract infection",B: Intentional contamination
2,"1. A 1-year-old girl is brought to a neurologist due to increasing seizure frequency over the past 2 months.\n2. She recently underwent a neurology evaluation which revealed hypsarrhythmia on electroencephalography (EEG) with a mix of slow waves, multifocal spikes, and asynchrony.\n3. Her parents have noticed the patient occasionally stiffens and spreads her arms at home.\n4. She was born at 38-weeks gestational age without complications.\n5. She has no other medical problems.\n6. Her medications consist of lamotrigine and valproic acid.\n7. Her temperature is 98.3°F (36.8°C), blood pressure is 90/75 mmHg, pulse is 94/min, and respirations are 22/min.\n8. Physical exam reveals innumerable hypopigmented macules on the skin and an irregularly shaped, thickened, and elevated plaque on the lower back.\n\nWhich of the following is most strongly associated with this patient's condition?\n\nA. Cardiac rhabdomyoma\nB. Glaucoma\nC. Optic glioma\nD. Polyostotic fibrous dysplasia",A: Cardiac rhabdomyoma
3,"1. A 17-year-old boy presents to his primary care physician with a chief concern of ""bad"" skin that has not improved despite home remedies.\n2. The patient has had lesions on his face that have persisted since he was 13 years of age.\n3. He has a diet high in refined carbohydrates and has gained 20 pounds since starting high school.\n4. Physical exam is notable for the findings in Figure A.\n5. The patient is started on benzoyl peroxide and topical retinoids.\n6. He returns 1 month later stating that his symptoms are roughly the same.\n\nWhich of the following is the most appropriate next step in management?\n\nA. Continue current therapy for 1 more month\nB. Dietary intervention\nC. Isoretinoin\nD. Topical antibiotics",D: Topical antibiotics
4,"1. A 55-year-old woman is brought to the emergency department by her husband with a 1 hour history of an unremitting headache.\n2. The headache started suddenly while she was 

In [34]:
# display(df_subset[["actual_question", "gpt_direct_prediction"]])

df_subset["answer_idx"] = df_op4.loc[df_subset.index, "answer_idx"]

df_subset["gpt_letter"] = df_subset["gpt4o_random1_direct_prediction"].str.strip().str[0]

df_subset["gpt_correct"] = df_subset.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_idx"] else "Incorrect",
    axis=1
)

correct_count = (df_subset["gpt_correct"] == "Correct").sum()
total_count = df_subset["gpt_correct"].notna().sum()
accuracy = correct_count / total_count

print(f"Correct predictions: {correct_count}")
print(f"Total predictions: {total_count}")
print(f"Accuracy: {accuracy:.2%}")


Correct predictions: 223
Total predictions: 298
Accuracy: 74.83%


## Randomize the Body of the Clinical Vignette

In [35]:
import random

def randomize_case_details(question_block):
    """
    Randomizes the lines of the clinical case except:
    - The first sentence (typically patient presentation)
    - The final question and answer options
    """
    # Split into lines and strip whitespace
    lines = [line.strip() for line in question_block.strip().split("\n") if line.strip()]
    
    # Find the index where the actual question begins (first line with '?')
    question_line_idx = next((i for i, line in enumerate(lines) if "?" in line), None)
    if question_line_idx is None:
        return question_block  # fail-safe: return original if no question found

    # First line: keep intact
    first_line = lines[0]

    # Lines to shuffle: lines between the first and the question line
    middle_lines = lines[1:question_line_idx]

    # Shuffle in-place
    random.shuffle(middle_lines)

    # Final block: reassemble all parts
    randomized_block = "\n".join([first_line] + middle_lines + lines[question_line_idx:])

    return randomized_block


In [36]:
df_subset = df_op4.head(5).copy()

df_op4["actual_question_randomized"] = df_op4["actual_question"].apply(randomize_case_details)
display(df_subset[["actual_question", "actual_question_randomized"]])

,actual_question,actual_question_randomized
0,"1. A 42-year-old woman is enrolled in a randomized controlled trial to study cardiac function in the setting of several different drugs.\n2. She is started on verapamil and instructed to exercise at 50% of her VO2 max while several cardiac parameters are being measured.\n\nDuring this experiment, which of the following represents the relative conduction speed through the heart from fastest to slowest?\n\nA. AV node > ventricles > atria > Purkinje fibers\nB. Purkinje fibers > ventricles > atria > AV node\nC. Purkinje fibers > atria > ventricles > AV node\nD. Purkinje fibers > AV node > ventricles > atria","1. A 42-year-old woman is enrolled in a randomized controlled trial to study cardiac function in the setting of several different drugs.\n2. She is started on verapamil and instructed to exercise at 50% of her VO2 max while several cardiac parameters are being measured.\nDuring this experiment, which of the following represents the relative conduction speed through the heart from fastest to slowest?\nA. AV node > ventricles > atria > Purkinje fibers\nB. Purkinje fibers > ventricles > atria > AV node\nC. Purkinje fibers > atria > ventricles > AV node\nD. Purkinje fibers > AV node > ventricles > atria"
1,"1. A 9-year-old girl presents to the emergency department with a fever and a change in her behavior.\n2. She presented with similar symptoms 6 weeks ago and was treated for an Escherchia coli infection.\n3. She also was treated for a urinary tract infection 10 weeks ago.\n4. Her mother says that last night her daughter felt ill, and her condition has been worsening.\n5. Her daughter experienced a severe headache and had a stiff neck.\n6. This morning she was minimally responsive, vomited several times, and produced a small amount of dark cloudy urine.\n7. The patient was born at 39 weeks and met all her developmental milestones.\n8. She is currently up to date on her vaccinations and did not have infections during early childhood.\n9. Her parents are divorced and her father has noted she does not seem to get sick when he takes care of her.\n10. Her temperature is 99.5°F (37.5°C), blood pressure is 60/35 mmHg, pulse is 190/min, respirations are 33/min, and oxygen saturation is 98% on room air.\n11. The patient is started on intravenous fluids, vasopressors, and broad-spectrum antibiotics.\n\nWhich of the following is the most appropriate underlying explanation for this patient's presentation?\n\nA. Gastroenteritis\nB. Intentional contamination\nC. Meningitis\nD. Urinary tract infection","1. A 9-year-old girl presents to the emergency department with a fever and a change in her behavior.\n8. She is currently up to date on her vaccinations and did not have infections during early childhood.\n3. She also was treated for a urinary tract infection 10 weeks ago.\n7. The patient was born at 39 weeks and met all her developmental milestones.\n6. This morning she was minimally responsive, vomited several times, and produced a small amount of dark cloudy urine.\n4. Her mother says that last night her daughter felt ill, and her condition has been worsening.\n2. She presented with similar symptoms 6 weeks ago and was treated for an Escherchia coli infection.\n10. Her temperature is 99.5°F (37.5°C), blood pressure is 60/35 mmHg, pulse is 190/min, respirations are 33/min, and oxygen saturation is 98% on room air.\n5. Her daughter experienced a severe headache and had a stiff neck.\n9. Her parents are divorced and her father has noted she does not seem to get sick when he takes care of her.\n11. The patient is started on intravenous fluids, vasopressors, and broad-spectrum antibiotics.\nWhich of the following is the most appropriate underlying explanation for this patient's presentation?\nA. Gastroenteritis\nB. Intentional contamination\nC. Meningitis\nD. Urinary tract infection"
2,"1. A 1-year-old girl is brought to a neurologist due to increasing seizure frequency over the past 2 months.\n2. Sh

In [37]:
# Apply GPT-4o to the randomized questions
df_op4["gpt4o_direct_prediction_randomized"] = df_op4["actual_question_randomized"].apply(generate_direct_prediction)

# Make sure long text is fully visible
import pandas as pd
pd.set_option("display.max_colwidth", None)

# Show a few examples with randomized prompt and prediction
display(df_op4[["actual_question_randomized", "gpt4o_direct_prediction_randomized"]].head())


,actual_question_randomized,gpt4o_direct_prediction_randomized
0,"1. A 42-year-old woman is enrolled in a randomized controlled trial to study cardiac function in the setting of several different drugs.\n2. She is started on verapamil and instructed to exercise at 50% of her VO2 max while several cardiac parameters are being measured.\nDuring this experiment, which of the following represents the relative conduction speed through the heart from fastest to slowest?\nA. AV node > ventricles > atria > Purkinje fibers\nB. Purkinje fibers > ventricles > atria > AV node\nC. Purkinje fibers > atria > ventricles > AV node\nD. Purkinje fibers > AV node > ventricles > atria",B: Purkinje fibers > ventricles > atria > AV node
1,"1. A 9-year-old girl presents to the emergency department with a fever and a change in her behavior.\n4. Her mother says that last night her daughter felt ill, and her condition has been worsening.\n5. Her daughter experienced a severe headache and had a stiff neck.\n2. She presented with similar symptoms 6 weeks ago and was treated for an Escherchia coli infection.\n7. The patient was born at 39 weeks and met all her developmental milestones.\n9. Her parents are divorced and her father has noted she does not seem to get sick when he takes care of her.\n11. The patient is started on intravenous fluids, vasopressors, and broad-spectrum antibiotics.\n3. She also was treated for a urinary tract infection 10 weeks ago.\n8. She is currently up to date on her vaccinations and did not have infections during early childhood.\n6. This morning she was minimally responsive, vomited several times, and produced a small amount of dark cloudy urine.\n10. Her temperature is 99.5°F (37.5°C), blood pressure is 60/35 mmHg, pulse is 190/min, respirations are 33/min, and oxygen saturation is 98% on room air.\nWhich of the following is the most appropriate underlying explanation for this patient's presentation?\nA. Gastroenteritis\nB. Intentional contamination\nC. Meningitis\nD. Urinary tract infection",B: Intentional contamination
2,"1. A 1-year-old girl is brought to a neurologist due to increasing seizure frequency over the past 2 months.\n7. Her temperature is 98.3°F (36.8°C), blood pressure is 90/75 mmHg, pulse is 94/min, and respirations are 22/min.\n2. She recently underwent a neurology evaluation which revealed hypsarrhythmia on electroencephalography (EEG) with a mix of slow waves, multifocal spikes, and asynchrony.\n3. Her parents have noticed the patient occasionally stiffens and spreads her arms at home.\n6. Her medications consist of lamotrigine and valproic acid.\n4. She was born at 38-weeks gestational age without complications.\n5. She has no other medical problems.\n8. Physical exam reveals innumerable hypopigmented macules on the skin and an irregularly shaped, thickened, and elevated plaque on the lower back.\nWhich of the following is most strongly associated with this patient's condition?\nA. Cardiac rhabdomyoma\nB. Glaucoma\nC. Optic glioma\nD. Polyostotic fibrous dysplasia",A: Cardiac rhabdomyoma
3,"1. A 17-year-old boy presents to his primary care physician with a chief concern of ""bad"" skin that has not improved despite home remedies.\n6. He returns 1 month later stating that his symptoms are roughly the same.\n3. He has a diet high in refined carbohydrates and has gained 20 pounds since starting high school.\n2. The patient has had lesions on his face that have persisted since he was 13 years of age.\n4. Physical exam is notable for the findings in Figure A.\n5. The patient is started on benzoyl peroxide and topical retinoids.\nWhich of the following is the most appropriate next step in management?\nA. Continue current therapy for 1 more month\nB. Dietary intervention\nC. Isoretinoin\nD. Topical antibiotics",A: Continue current therapy for 1 more month
4,"1. A 55-year-old woman is brought to the emergency department by her husband with a 1 hour history of an unremitting headache.\n5. She also develops nausea

In [38]:
# Create a new column with just the predicted answer letter (A/B/C/D)
df_op4["gpt4o_letter_randomized"] = df_op4["gpt4o_direct_prediction_randomized"].str.strip().str[0]

# Compare predictions with actual answers
df_op4["gpt4o_correct_randomized"] = df_op4.apply(
    lambda row: "Correct" if row["gpt4o_letter_randomized"] == row["answer_idx"] else "Incorrect",
    axis=1
)

# Accuracy stats
correct_count = (df_op4["gpt4o_correct_randomized"] == "Correct").sum()
total_count = df_op4["gpt4o_correct_randomized"].notna().sum()
accuracy = correct_count / total_count

# Print stats
print(f"Correct predictions: {correct_count}")
print(f"Total predictions: {total_count}")
print(f"Accuracy: {accuracy:.2%}")


Correct predictions: 217
Total predictions: 298
Accuracy: 72.82%


In [39]:
df_op4[[
    "gpt4o_letter_randomized", 
    "gpt4o_correct_randomized"
]].to_csv("gpt4o_randomized_predictions.csv", index=False)


In [31]:
df_op4[[
    "gpt_letter_randomized", 
    "gpt_correct_randomized"
]].to_csv("o3_randomized_predictions.csv", index=False)


## W/ Reasoning

In [9]:
def get_reasoning_and_answer(bullet_question):
    prompt = f"""
You are a highly capable and careful clinical reasoning assistant. Given a clinical vignette with multiple-choice options, think step by step to identify the most likely correct answer.

{bullet_question}

First, explain your reasoning process in detail as if you were walking a student through the clinical logic step by step.

Then, at the end, write your final answer on a new line in the format:
Final Answer: <number>

Only use one of the digits 0, 1, 2, or 3 to indicate your final choice.
"""

    try:
        client = openai.OpenAI()

        response = client.chat.completions.create(
            model="o3-mini",
            messages=[{"role": "user", "content": prompt}]
        )

        content = response.choices[0].message.content.strip()

        if "Final Answer:" in content:
            parts = content.rsplit("Final Answer:", 1)
            reasoning = parts[0].strip()
            # Protect against empty answer part
            answer_part = parts[1].strip()
            answer = answer_part.split()[0] if answer_part else ""
        else:
            reasoning = content
            answer = ""

        return reasoning, answer

    except Exception as e:
        return "Error during generation", ""


In [11]:
import pandas as pd
from tqdm.notebook import tqdm
import openai

# Setup: result file path and loading progress
results_path = "medbullets_o3_reasoning_results.csv"

try:
    results_df = pd.read_csv(results_path, index_col=0)
    done_indices = set(results_df.index)
    print(f"Resuming from {len(done_indices)} previously completed rows.")
except FileNotFoundError:
    results_df = pd.DataFrame(columns=["o3_reasoning_process", "o3_reasoning_answer"])
    done_indices = set()
    print("Starting fresh.")

# Prediction loop with periodic saving
new_rows = []

for idx in tqdm(df_op4.index):
    if idx in done_indices:
        continue

    bullet_question = df_op4.at[idx, "bullet_question"]

    try:
        reasoning, answer = get_reasoning_and_answer(bullet_question)
    except Exception as e:
        print(f"Error at index {idx}: {e}")
        reasoning, answer = None, None

    new_rows.append((idx, reasoning, answer))

    if len(new_rows) >= 10:  # Save every 10 rows
        temp_df = pd.DataFrame(new_rows, columns=["index", "o3_reasoning_process", "o3_reasoning_answer"])
        temp_df.set_index("index", inplace=True)
        results_df = pd.concat([results_df, temp_df])
        results_df.to_csv(results_path)
        new_rows = []

# Final save if remaining
if new_rows:
    temp_df = pd.DataFrame(new_rows, columns=["index", "o3_reasoning_process", "o3_reasoning_answer"])
    temp_df.set_index("index", inplace=True)
    results_df = pd.concat([results_df, temp_df])
    results_df.to_csv(results_path)

# Merge results into main DataFrame
df_op4 = df_op4.join(results_df, how="left", rsuffix="_new")

# Save full merged output
df_op4.to_csv("medbullets_o3_reasoning_results.csv", index=False)


Starting fresh.


  0%|          | 0/298 [00:00<?, ?it/s]

In [19]:
# Define numeric-to-letter mapping
index_to_letter = {"0": "A", "1": "B", "2": "C", "3": "D"}

# Extract digit, get first column from extract result, then map to A–D
df_op4["o3_reasoning_answer"] = (
    df_op4["o3_reasoning_answer"]
    .astype(str)
    .str.extract(r'(\d)')[0]  # <-- get the first column from the DataFrame
    .map(index_to_letter)
)

# Clean the gold answer column
df_op4["answer_idx"] = df_op4["answer_idx"].astype(str).str.strip().str.upper()

# Compute accuracy
if "o3_reasoning_answer" in df_op4.columns and "answer_idx" in df_op4.columns:
    total = len(df_op4)
    correct = (df_op4["o3_reasoning_answer"] == df_op4["answer_idx"]).sum()
    accuracy = (correct / total) * 100 if total > 0 else 0.0

    print(f"Total Evaluated Examples: {total}")
    print(f"Correct Predictions: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")
else:
    print("Missing required columns: 'o3_reasoning_answer' and/or 'answer_idx'. Evaluation aborted.")


Total Evaluated Examples: 298
Correct Predictions: 255
Accuracy: 85.57%


## 4o

In [22]:
def get_reasoning_and_answer(bullet_question):
    prompt = f"""
You are a highly capable and careful clinical reasoning assistant. Given a clinical vignette with multiple-choice options, think step by step to identify the most likely correct answer.

{bullet_question}

First, explain your reasoning process in detail as if you were walking a student through the clinical logic step by step.

Then, at the end, write your final answer on a new line in the format:
Final Answer: <number>

Only use one of the digits 0, 1, 2, or 3 to indicate your final choice.
"""

    try:
        client = openai.OpenAI()

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature = 0
        )

        content = response.choices[0].message.content.strip()

        if "Final Answer:" in content:
            parts = content.rsplit("Final Answer:", 1)
            reasoning = parts[0].strip()
            # Protect against empty answer part
            answer_part = parts[1].strip()
            answer = answer_part.split()[0] if answer_part else ""
        else:
            reasoning = content
            answer = ""

        return reasoning, answer

    except Exception as e:
        return "Error during generation", ""


In [23]:
import pandas as pd
from tqdm.notebook import tqdm
import openai

# Path to save intermediate and final results
results_path = "medbullets_4o_reasoning_results.csv"

# Try loading previous results to resume from where it left off
try:
    results_df = pd.read_csv(results_path, index_col=0)
    done_indices = set(results_df.index)
    print(f"Resuming from {len(done_indices)} previously completed rows.")
except FileNotFoundError:
    results_df = pd.DataFrame(columns=["4o_reasoning_process", "4o_reasoning_answer"])
    done_indices = set()
    print("Starting fresh.")

# Collect new results in batches
new_rows = []

for idx in tqdm(df_op4.index):
    if idx in done_indices:
        continue

    bullet_question = df_op4.at[idx, "bullet_question"]

    try:
        reasoning, answer = get_reasoning_and_answer(bullet_question)
    except Exception as e:
        print(f"Error at index {idx}: {e}")
        reasoning, answer = "Error during generation", ""

    new_rows.append((idx, reasoning, answer))

    if len(new_rows) >= 10:
        temp_df = pd.DataFrame(new_rows, columns=["index", "4o_reasoning_process", "4o_reasoning_answer"])
        temp_df.set_index("index", inplace=True)
        results_df = pd.concat([results_df, temp_df])
        results_df.to_csv(results_path)
        new_rows = []

# Save any remaining predictions
if new_rows:
    temp_df = pd.DataFrame(new_rows, columns=["index", "4o_reasoning_process", "4o_reasoning_answer"])
    temp_df.set_index("index", inplace=True)
    results_df = pd.concat([results_df, temp_df])
    results_df.to_csv(results_path)

# Merge predictions back into main dataset
df_op4 = df_op4.join(results_df, how="left", rsuffix="_new")

# Save full merged DataFrame
df_op4.to_csv("medbullets_4o_reasoning_results.csv", index=False)


Starting fresh.


  0%|          | 0/298 [00:00<?, ?it/s]

In [24]:
# Define numeric-to-letter mapping
index_to_letter = {"0": "A", "1": "B", "2": "C", "3": "D"}

# Extract digit from '4o_reasoning_answer' and map it to A–D
df_op4["4o_reasoning_answer"] = (
    df_op4["4o_reasoning_answer"]
    .astype(str)
    .str.extract(r'(\d)')[0]  # Take the first (and only) capture group
    .map(index_to_letter)
)

# Clean the reference answer column
df_op4["answer_idx"] = df_op4["answer_idx"].astype(str).str.strip().str.upper()

# Compute and display accuracy
if "4o_reasoning_answer" in df_op4.columns and "answer_idx" in df_op4.columns:
    total = len(df_op4)
    correct = (df_op4["4o_reasoning_answer"] == df_op4["answer_idx"]).sum()
    accuracy = (correct / total) * 100 if total > 0 else 0.0

    print(f"Total Evaluated Examples: {total}")
    print(f"Correct Predictions: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")
else:
    print("Missing required columns: '4o_reasoning_answer' and/or 'answer_idx'. Evaluation aborted.")


Total Evaluated Examples: 298
Correct Predictions: 238
Accuracy: 79.87%


## Llama Results Direct Prediction

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
from huggingface_hub import login

In [71]:
# Load Llama 3.1 7B Instruct model and tokenizer
model_name = "meta-llama/Llama-3.1-8B-Instruct"
hf_token = os.getenv("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    token=hf_token
)

# Verify Model Loading
print("Model and tokenizer loaded successfully!")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model and tokenizer loaded successfully!


In [72]:
def score_option_from_prompt(prompt, option_text):
    """
    Scores a full prompt + specific answer option using the model's log-likelihood.
    """
    full_prompt = f"{prompt}\nAnswer: {option_text}"
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return -outputs.loss.item()  # higher is better


In [73]:
def get_best_answer_scored(row):
    prompt = row["actual_question"]

    # Score each choice independently
    scores = {
        "A": score_option_from_prompt(prompt, row["opa"]),
        "B": score_option_from_prompt(prompt, row["opb"]),
        "C": score_option_from_prompt(prompt, row["opc"]),
        "D": score_option_from_prompt(prompt, row["opd"])
    }

    best_letter = max(scores, key=scores.get)
    best_text = {
        "A": row["opa"],
        "B": row["opb"],
        "C": row["opc"],
        "D": row["opd"]
    }[best_letter]

    return pd.Series([best_letter, best_text])


In [75]:
import pandas as pd

# Make sure all content is shown (especially for long clinical questions/answers)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

# Show selected columns
display(df_subset[["actual_question", "Llama_letter", "Llama_answer"]])


,actual_question,Llama_letter,Llama_answer
0,"- A 42-year-old woman is enrolled in a randomized controlled trial to study cardiac function in the setting of several different drugs.\n- She is started on verapamil and instructed to exercise at 50% of her VO2 max while several cardiac parameters are being measured.\n\nDuring this experiment, which of the following represents the relative conduction speed through the heart from fastest to slowest?\n\n- A: AV node > ventricles > atria > Purkinje fibers\n- B: Purkinje fibers > ventricles > atria > AV node\n- C: Purkinje fibers > atria > ventricles > AV node\n- D: Purkinje fibers > AV node > ventricles > atria",C,Purkinje fibers > atria > ventricles > AV node
1,"- A 9-year-old girl presents to the emergency department with a fever and a change in her behavior.\n- She presented with similar symptoms 6 weeks ago and was treated for an Escherchia coli infection.\n- She also was treated for a urinary tract infection 10 weeks ago.\n- Her mother says that last night her daughter felt ill, and her condition has been worsening.\n- Her daughter experienced a severe headache and had a stiff neck.\n- This morning she was minimally responsive, vomited several times, and produced a small amount of dark cloudy urine.\n- The patient was born at 39 weeks and met all her developmental milestones.\n- She is currently up to date on her vaccinations and did not have infections during early childhood.\n- Her parents are divorced and her father has noted she does not seem to get sick when he takes care of her.\n- Her temperature is 99.5°F (37.5°C), blood pressure is 60/35 mmHg, pulse is 190/min, respirations are 33/min, and oxygen saturation is 98% on room air.\n- The patient is started on intravenous fluids, vasopressors, and broad-spectrum antibiotics.\n\nWhich of the following is the most appropriate underlying explanation for this patient's presentation?\n\n- A: Gastroenteritis\n- B: Intentional contamination\n- C: Meningitis\n- D: Urinary tract infection",C,Meningitis
2,"- A 1-year-old girl is brought to a neurologist due to increasing seizure frequency over the past 2 months.\n- She recently underwent a neurology evaluation which revealed hypsarrhythmia on electroencephalography (EEG) with a mix of slow waves, multifocal spikes, and asynchrony.\n- Her parents have noticed the patient occasionally stiffens and spreads her arms at home.\n- She was born at 38-weeks gestational age without complications.\n- She has no other medical problems.\n- Her medications consist of lamotrigine and valproic acid.\n- Her temperature is 98.3°F (36.8°C), blood pressure is 90/75 mmHg, pulse is 94/min, and respirations are 22/min.\n- Physical exam reveals innumerable hypopigmented macules on the skin and an irregularly shaped, thickened, and elevated plaque on the lower back.\n\nWhich of the following is most strongly associated with this patient's condition?\n\n- A: Cardiac rhabdomyoma\n- B: Glaucoma\n- C: Optic glioma\n- D: Polyostotic fibrous dysplasia",C,Optic glioma
3,"- A 17-year-old boy presents to his primary care physician with a chief concern of ""bad"" skin that has not improved despite home remedies.\n- The patient has had lesions on his face that have persisted since he was 13 years of age.\n- He has a diet high in refined carbohydrates and has gained 20 pounds since starting high school.\n- Physical exam is notable for the findings in Figure A.\n- The patient is started on benzoyl peroxide and topical retinoids.\n- He returns 1 month later stating that his symptoms are roughly the same.\n\nWhich of the following is the most appropriate next step in management?\n\n- A: Continue current therapy for 1 more month\n- B: Dietary intervention\n- C: Isoretinoin\n- D: Topical antibiotics",C,Isoretinoin
4,"- A 55-year-old woman is brought to the emergency department by her husband with a 1 hour history of an unremitting headache.\n- The headache started suddenly while she was eating dinner and she says it feels lik